In [ ]:
import pandas as pd
import kagglehub
import ast
import numpy as np
from collections import Counter
from unidecode import unidecode
import pycountry

In [ ]:
27753444/283228

In [ ]:
path_ml = kagglehub.dataset_download("grouplens/movielens-latest-full")
path_tmdb = kagglehub.dataset_download("rounakbanik/the-movies-dataset")
print("MovieLens path:", path_ml)
print("TMDB path:", path_tmdb)

# MovieLens files
ratings_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/ratings.csv')
movies_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/movies.csv')
links_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/links.csv')
tags_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/tags.csv')

# TMDB files
metadata_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/movies_metadata.csv', low_memory=False)
credits_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/credits.csv')
keywords_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/keywords.csv')

In [ ]:
def get_director(crew):
    return next((p['name'] for p in crew if p.get('job') == 'Director'), None)

def get_producer(crew):
    return next((p['name'] for p in crew if p.get('job') == 'Producer'), None)

def get_lead_actor(cast):
    return next((p['name'] for p in sorted(cast, key=lambda x: x.get('order', float('inf'))) if 'name' in p), None)

def get_gender_of_lead(cast, lead_name):
    return next((p['gender'] for p in cast if p.get('name') == lead_name), None)

def get_other_lead(cast, lead_name):
    lead_gender = get_gender_of_lead(cast, lead_name)
    opposite_gender = {1: 2, 2: 1}.get(lead_gender)
    return next(
        (p['name'] for p in sorted(cast, key=lambda x: x.get('order', float('inf')))
         if p.get('name') != lead_name and p.get('gender') == opposite_gender),
        None
    )

def get_other_actors(cast, exclude_names, max_count=3):
    return [
        p['name'] for p in sorted(cast, key=lambda x: x.get('order', float('inf')))
        if p.get('name') not in exclude_names and p.get('name') is not None
    ][:max_count]


In [ ]:
def iso_639_to_name(code):
    try:
        return pycountry.languages.get(alpha_2=code).name
    except:
        return code.upper()

def safe_language_name(d):
    if isinstance(d, dict):
        name = d.get('name')
        code = d.get('iso_639_1')
        if name and '?' not in name:
            return unidecode(name)
        if code:
            return iso_639_to_name(code)
    return None


In [ ]:
# Clean links
links_ml = links_ml[links_ml['tmdbId'].notnull()]
links_ml['tmdbId'] = links_ml['tmdbId'].astype(int)

# Clean metadata
metadata_tmdb = metadata_tmdb[pd.to_numeric(metadata_tmdb['id'], errors='coerce').notnull()]
metadata_tmdb['id'] = metadata_tmdb['id'].astype(int)
metadata_tmdb['genres'] = metadata_tmdb['genres'].fillna('[]').apply(ast.literal_eval)

# Parse credits
credits_tmdb['cast'] = credits_tmdb['cast'].apply(ast.literal_eval)
credits_tmdb['crew'] = credits_tmdb['crew'].apply(ast.literal_eval)
credits_tmdb['tmdbId'] = credits_tmdb['id']
credits_tmdb['director'] = credits_tmdb['crew'].apply(get_director)
credits_tmdb['producer'] = credits_tmdb['crew'].apply(get_producer)
credits_tmdb['lead_actor'] = credits_tmdb['cast'].apply(get_lead_actor)
credits_tmdb['other_lead'] = credits_tmdb.apply(
    lambda row: get_other_lead(row['cast'], row['lead_actor']), axis=1
)
credits_tmdb['other_actors'] = credits_tmdb.apply(
    lambda row: get_other_actors(row['cast'], exclude_names={row['lead_actor'], row['other_lead']}), axis=1
)

# Parse keywords
keywords_tmdb['keywords'] = keywords_tmdb['keywords'].fillna('[]').apply(ast.literal_eval)
keywords_tmdb['keywords'] = keywords_tmdb['keywords'].apply(lambda x: [d['name'] for d in x if isinstance(d, dict)])
keywords_tmdb['tmdbId'] = keywords_tmdb['id']

# Aggregate tags
tags_agg = tags_ml.groupby('movieId')['tag'].apply(lambda x: list(set(x))).reset_index()

# Ratings stats
rating_stats = ratings_ml.groupby('movieId')['rating'].agg(['mean', 'min', 'max', 'count']).reset_index()
rating_stats.columns = ['movieId', 'vote_average', 'vote_min', 'vote_max', 'vote_count']

In [ ]:
movies_ml_links = pd.merge(movies_ml, links_ml, on='movieId')
metadata_tmdb = metadata_tmdb.rename(columns={'id': 'tmdbId'})
movies_full = pd.merge(movies_ml_links, metadata_tmdb, on='tmdbId', how='inner')

movies_full = pd.merge(movies_full, credits_tmdb[['tmdbId', 'director', 'producer', 'other_lead', 'other_actors','lead_actor']], on='tmdbId', how='left')
movies_full = pd.merge(movies_full, keywords_tmdb[['tmdbId', 'keywords']], on='tmdbId', how='left')
movies_full = pd.merge(movies_full, tags_agg, on='movieId', how='left')
movies_full = pd.merge(movies_full, rating_stats, on='movieId', how='left')

In [ ]:
# Runtime binning
bin_edges = list(range(0, 301, 30)) + [np.inf]
labels = [f'{bin_edges[i]}–{bin_edges[i+1]}min' if bin_edges[i+1] != np.inf else f'{bin_edges[i]}min+' for i in range(len(bin_edges) - 1)]
movies_full['runtime_bin'] = pd.cut(movies_full['runtime'], bins=bin_edges, labels=labels)

# Release dates
movies_full['release_date_parsed'] = pd.to_datetime(movies_full['release_date'], errors='coerce')
release_year_tmdb = movies_full['release_date_parsed'].dt.year
release_year_ml = movies_full['title_x'].str.extract(r'\((\d{4})\)')[0].astype(float)

movies_full['release_year_tmdb'] = release_year_tmdb
movies_full['release_year_ml'] = release_year_ml
movies_full['release_year'] = release_year_tmdb.combine_first(release_year_ml)
movies_full['release_year_merged'] = movies_full[['release_year_tmdb', 'release_year_ml']].min(axis=1)

# Ratings
movies_full['vote_count'] = movies_full[['vote_count_x', 'vote_count_y']].max(axis=1)
movies_full['vote_average'] = movies_full.apply(
    lambda row: row['vote_average_x'] if row['vote_count_x'] >= row['vote_count_y'] else row['vote_average_y'],
    axis=1
)

# Title
movies_full['title'] = movies_full['title_y'].combine_first(movies_full['title_x'])

# Genres
movies_full['genres_x_list'] = movies_full['genres_x'].fillna('').apply(lambda x: x.split('|') if isinstance(x, str) else [])
movies_full['genres_y_list'] = movies_full['genres_y'].apply(lambda x: [d['name'] for d in x if isinstance(d, dict)] if isinstance(x, list) else [])
movies_full['genre_list'] = movies_full.apply(lambda row: sorted(set(row['genres_x_list']) | set(row['genres_y_list'])), axis=1)

# Main genre (prefer TMDB, fallback to MovieLens)
def extract_main_genre(genres_y): return genres_y[0]['name'] if isinstance(genres_y, list) and len(genres_y) > 0 and isinstance(genres_y[0], dict) else None
movies_full['main_genre'] = movies_full['genres_y'].apply(extract_main_genre)
movies_full['main_genre'] = movies_full.apply(lambda row: row['main_genre'] if pd.notnull(row['main_genre']) else (row['genres_x_list'][0] if row['genres_x_list'] else None), axis=1)

# Additional features
movies_full['release_month'] = movies_full['release_date_parsed'].dt.month
movies_full['release_decade'] = (movies_full['release_year'] // 10) * 10
movies_full['popularity_score'] = movies_full['vote_average'] * np.log1p(movies_full['vote_count'])

In [ ]:
all_genres = movies_full['genre_list'].explode()
genre_counts = Counter(all_genres)
genre_count_df = pd.DataFrame(genre_counts.items(), columns=['genre', 'count']).sort_values(by='count', ascending=False)
genre_count_df.head(10)  # Top 10 genres

In [ ]:
movies_full = movies_full.drop(columns=[
    # 'title_x', 'title_y',
    # 'vote_average_x', 'vote_average_y',
    # 'vote_count_x', 'vote_count_y',
    # 'release_year_tmdb', 'release_year_ml',
    'release_date',
    # 'genres_x', 'genres_y'
])

movies_full = movies_full.rename(columns={'release_date_parsed': 'release_date'})

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
movies_full["production_countries"][3]

In [ ]:
movies_full["production_countries"].unique()

In [ ]:
movies_full.columns

In [ ]:
movies_full["spoken_languages"].unique()

In [ ]:
movies_full['production_countries_parsed'] = movies_full['production_countries'].apply(
    lambda x: ast.literal_eval(x) if pd.notnull(x) else []
)

movies_full['spoken_languages_parsed'] = movies_full['spoken_languages'].apply(
    lambda x: ast.literal_eval(x) if pd.notnull(x) else []
)

# movies_full['production_country_names'] = movies_full['production_countries_parsed'].apply(
#     lambda lst: [d['name'] for d in lst if isinstance(d, dict) and 'name' in d]
# )

# movies_full['spoken_language_names'] = movies_full['spoken_languages_parsed'].apply(
#     lambda lst: [d['name'] for d in lst if isinstance(d, dict) and 'name' in d]
# )

movies_full['production_country_names'] = movies_full['production_countries_parsed'].apply(
    lambda lst: [unidecode(d['name']) for d in lst if isinstance(d, dict) and 'name' in d]
)

# Fix: spoken_language_names
movies_full['spoken_language_names'] = movies_full['spoken_languages_parsed'].apply(
    lambda lst: [safe_language_name(d) for d in lst if isinstance(d, dict)]
)

In [ ]:
# movies_full['main_language'] = movies_full['spoken_language_names'].apply(
#     lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
# )
# movies_full['main_country'] = movies_full['production_country_names'].apply(
#     lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
# )

movies_full['main_language'] = movies_full['spoken_language_names'].apply(
    lambda x: unidecode(x[0]) if isinstance(x, list) and len(x) > 0 else None
)

# Main production country with transliteration
movies_full['main_country'] = movies_full['production_country_names'].apply(
    lambda x: unidecode(x[0]) if isinstance(x, list) and len(x) > 0 else None
)

In [ ]:
# Show a few raw entries
movies_full[movies_full['main_country'].str.contains(r'\?{2,}', na=False)][['title', 'spoken_language_names']]


In [ ]:
movies_full['has_translation'] = movies_full.apply(
    lambda row: [lang for lang in row['spoken_language_names'] if lang != row['main_language']]
    if isinstance(row['spoken_language_names'], list) else [],
    axis=1
)

movies_full['has_translation']

In [ ]:
movies_full.iloc[0]["belongs_to_collection"]

In [ ]:
movies_full['collection_name'] = movies_full['belongs_to_collection'].apply(
    lambda x: ast.literal_eval(x).get('name') if pd.notnull(x) else None
)
movies_full.loc[movies_full['collection_name'].notnull(), ['title', 'collection_name']].head()

In [ ]:
movies_full.columns

In [ ]:

movies_full.iloc[0]["production_companies"]


In [ ]:
movies_full['production_companies_parsed'] = movies_full['production_companies'].apply(
    lambda x: ast.literal_eval(x) if pd.notnull(x) else []
)

# Step 2: Extract list of company names
movies_full['production_company_names'] = movies_full['production_companies_parsed'].apply(
    lambda lst: [d['name'] for d in lst if isinstance(d, dict) and 'name' in d]
)

# Step 3: Get main/first company
movies_full['main_production_company'] = movies_full['production_company_names'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
)

In [ ]:
movies_full['production_company_names']

In [ ]:
movies_full.columns

In [ ]:
movies_full

In [ ]:
movies_full.columns

In [ ]:
movies_full[["poster_path"]]

In [ ]:
IMAGE_BASE_URL = "https://image.tmdb.org/t/p/w500"

# Add new column with full URL
movies_full['poster_url'] = movies_full['poster_path'].apply(
    lambda path: f"{IMAGE_BASE_URL}{path}" if pd.notnull(path) else None
)

In [ ]:
movies_full['is_short_film'] = movies_full['runtime'] < 40
movies_full['is_feature_film'] = (movies_full['runtime'] >= 40) & (movies_full['runtime'] < 120)
movies_full['is_long_film'] = movies_full['runtime'] >= 120

movies_full['movie_age'] = 2025 - movies_full['release_year']

movies_full['keyword_count'] = movies_full['keywords'].apply(lambda x: len(x) if isinstance(x, list) else 0)



In [ ]:
movies_full

In [ ]:
actor_popularity = (
    movies_full.groupby("lead_actor")
    .agg({
        "popularity_score": "sum",
        "vote_average": "mean",
        "vote_count": "sum",
        "revenue": "sum"
    })
    .fillna(0)
)

# Normalize scores (optional, for fair scaling)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
actor_popularity_scaled = pd.DataFrame(
    scaler.fit_transform(actor_popularity),
    columns=actor_popularity.columns,
    index=actor_popularity.index
)

# Combine weighted popularity
actor_popularity_scaled["popularity_index"] = (
    0.4 * actor_popularity_scaled["popularity_score"] +
    0.3 * actor_popularity_scaled["vote_average"] +
    0.2 * actor_popularity_scaled["vote_count"] +
    0.1 * actor_popularity_scaled["revenue"]
)

# Map back to DataFrame
movies_full["lead_actor_popularity"] = movies_full["lead_actor"].map(actor_popularity_scaled["popularity_index"])

actor_popularity = (
    movies_full.groupby("other_lead")
    .agg({
        "popularity_score": "sum",
        "vote_average": "mean",
        "vote_count": "sum",
        "revenue": "sum"
    })
    .fillna(0)
)

# Normalize scores (optional, for fair scaling)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
actor_popularity_scaled = pd.DataFrame(
    scaler.fit_transform(actor_popularity),
    columns=actor_popularity.columns,
    index=actor_popularity.index
)

# Combine weighted popularity
actor_popularity_scaled["popularity_index"] = (
    0.4 * actor_popularity_scaled["popularity_score"] +
    0.3 * actor_popularity_scaled["vote_average"] +
    0.2 * actor_popularity_scaled["vote_count"] +
    0.1 * actor_popularity_scaled["revenue"]
)

# Map back to DataFrame
movies_full["other_lead_popularity"] = movies_full["lead_actor"].map(actor_popularity_scaled["popularity_index"])



In [ ]:
actor_genres = movies_full.groupby("lead_actor")["genre_list"].apply(
    lambda genres: len(set(g for sublist in genres for g in sublist))
)
movies_full["lead_actor_genre_diversity"] = movies_full["lead_actor"].map(actor_genres)

actor_genres = movies_full.groupby("other_lead")["genre_list"].apply(
    lambda genres: len(set(g for sublist in genres for g in sublist))
)
movies_full["other_lead_genre_diversity"] = movies_full["other_lead"].map(actor_genres)


In [ ]:
# Group by director
director_popularity = (
    movies_full.groupby("director")
    .agg({
        "popularity_score": "sum",
        "vote_average": "mean",
        "vote_count": "sum",
        "revenue": "sum"
    })
    .fillna(0)
)


# Group by producer
producer_popularity = (
    movies_full.groupby("producer")
    .agg({
        "popularity_score": "sum",
        "vote_average": "mean",
        "vote_count": "sum",
        "revenue": "sum"
    })
    .fillna(0)
)


director_scaled = pd.DataFrame(
    MinMaxScaler().fit_transform(director_popularity),
    columns=director_popularity.columns,
    index=director_popularity.index
)
director_scaled["popularity_index"] = (
    0.4 * director_scaled["popularity_score"] +
    0.3 * director_scaled["vote_average"] +
    0.2 * director_scaled["vote_count"] +
    0.1 * director_scaled["revenue"]
)

# Normalize producer scores
producer_scaled = pd.DataFrame(
    MinMaxScaler().fit_transform(producer_popularity),
    columns=producer_popularity.columns,
    index=producer_popularity.index
)
producer_scaled["popularity_index"] = (
    0.4 * producer_scaled["popularity_score"] +
    0.3 * producer_scaled["vote_average"] +
    0.2 * producer_scaled["vote_count"] +
    0.1 * producer_scaled["revenue"]
)

director_scaled = pd.DataFrame(
    MinMaxScaler().fit_transform(director_popularity),
    columns=director_popularity.columns,
    index=director_popularity.index
)
director_scaled["popularity_index"] = (
    0.4 * director_scaled["popularity_score"] +
    0.3 * director_scaled["vote_average"] +
    0.2 * director_scaled["vote_count"] +
    0.1 * director_scaled["revenue"]
)

# Normalize producer scores
producer_scaled = pd.DataFrame(
    MinMaxScaler().fit_transform(producer_popularity),
    columns=producer_popularity.columns,
    index=producer_popularity.index
)
producer_scaled["popularity_index"] = (
    0.4 * producer_scaled["popularity_score"] +
    0.3 * producer_scaled["vote_average"] +
    0.2 * producer_scaled["vote_count"] +
    0.1 * producer_scaled["revenue"]
)

In [ ]:
# Group by production company
company_popularity = (
    movies_full.groupby("main_production_company")
    .agg({
        "popularity_score": "sum",
        "vote_average": "mean",
        "vote_count": "sum",
        "revenue": "sum"
    })
    .fillna(0)
)

# Normalize
company_scaled = pd.DataFrame(
    MinMaxScaler().fit_transform(company_popularity),
    columns=company_popularity.columns,
    index=company_popularity.index
)

# Weighted popularity index
company_scaled["popularity_index"] = (
    0.4 * company_scaled["popularity_score"] +
    0.3 * company_scaled["vote_average"] +
    0.2 * company_scaled["vote_count"] +
    0.1 * company_scaled["revenue"]
)

# Map to movies_full
movies_full["company_popularity"] = movies_full["main_production_company"].map(company_scaled["popularity_index"])


In [ ]:
# Group by collection
collection_strength = (
    movies_full[movies_full["collection_name"].notna()]
    .groupby("collection_name")
    .agg({
        "popularity_score": "sum",
        "vote_average": "mean",
        "vote_count": "sum",
        "revenue": "sum"
    })
    .fillna(0)
)

# Normalize
collection_scaled = pd.DataFrame(
    MinMaxScaler().fit_transform(collection_strength),
    columns=collection_strength.columns,
    index=collection_strength.index
)

collection_scaled["franchise_strength"] = (
    0.4 * collection_scaled["popularity_score"] +
    0.3 * collection_scaled["vote_average"] +
    0.2 * collection_scaled["vote_count"] +
    0.1 * collection_scaled["revenue"]
)

# Map to movies_full
movies_full["franchise_strength"] = movies_full["collection_name"].map(collection_scaled["franchise_strength"])


In [ ]:
movies_full["budget"] = pd.to_numeric(movies_full["budget"], errors="coerce")
movies_full["revenue"] = pd.to_numeric(movies_full["revenue"], errors="coerce")

# Now compute profit and ROI safely
movies_full["profit"] = movies_full["revenue"] - movies_full["budget"]
movies_full["roi"] = movies_full.apply(
    lambda row: row["profit"] / row["budget"] if pd.notnull(row["budget"]) and row["budget"] > 0 else None,
    axis=1
)

In [ ]:
movies_full["genre_count"] = movies_full["genre_list"].apply(len)
movies_full["keyword_count"] = movies_full["keywords"].apply(lambda x: len(x) if isinstance(x, list) else 0)
movies_full["tag_count"] = movies_full["tag"].apply(lambda x: len(x) if isinstance(x, list) else 0)
movies_full["is_recent"] = movies_full["release_year"].apply(lambda y: 1 if y and y >= 2015 else 0)

movies_full["critical_success"] = movies_full[["vote_average", "popularity_score"]].mean(axis=1)
movies_full["crowd_approval"] = movies_full["vote_average"] * np.log1p(movies_full["vote_count"])


In [ ]:
movies_full["log_budget"] = np.log1p(movies_full["budget"])
movies_full["log_revenue"] = np.log1p(movies_full["revenue"])
movies_full["log_profit"] = np.log1p(movies_full["profit"])


In [ ]:
for role in ["lead_actor", "director", "producer"]:
    role_counts = movies_full[role].value_counts()
    score = movies_full[role].map(role_counts)
    scaler = MinMaxScaler()
    movies_full[f"{role}_popularity"] = scaler.fit_transform(score.values.reshape(-1, 1))


In [ ]:
# movies_full = movies_full[[
#     # 1. Movie & genre
#     'title', 'main_genre', 'genre_list', 'keywords', 'tag', 'collection_name',

#     # 2. Cast & crew
#     'director', 'producer', 'lead_actor', 'other_lead',
#     'other_actors',

#     # 3. Production
#     'main_production_company', 'production_company_names',
#     'main_country', 'production_country_names',

#     # 4. Language
#     'main_language', 'spoken_language_names', 'has_translation',

#     # 5. Temporal
#     'release_date', 'release_year', 'release_month', 'release_decade',
#     'runtime', 'runtime_bin',

#     # 6. Ratings
#     'vote_average', 'vote_count', 'vote_min', 'vote_max', 'popularity_score',

#     # 7. Financials
#     'budget', 'revenue',

#     # 8. IDs
#     'movieId', 'imdbId', 'tmdbId', 'imdb_id'
# ]]


In [ ]:
[x for x in movies_full.columns if (("_x" in x) or ("_y" in x)) and x != "release_year"]

In [ ]:
movies_full = movies_full.drop(columns=[x for x in movies_full.columns if (("_x" in x) or ("_y" in x)) and x != "release_year"])

In [ ]:
[x for x in movies_full.columns if ("_parsed" in x)]

In [ ]:
movies_full = movies_full.drop(columns=[x for x in movies_full.columns if ("_parsed" in x)])
movies_full = movies_full.drop(columns=[x for x in movies_full.columns if (x == "original_title" or x == "video" or 
                                                                           x == "poster_path" or 
                                                                           x == "belongs_to_collection" or
                                                                           x == "spoken_languages" or
                                                                           x == "production_countries" or
                                                                           x == "production_companies"
                                                                           )])


In [ ]:
len(movies_full.columns)
# 71

In [ ]:
movies_full.columns

In [ ]:
final_column_order = [
    # Identifiers (IDs)
    'movieId',
    'imdbId',
    'tmdbId',
    'imdb_id', # Included as it was in your list

    # Core Movie Info (Titles, Overview, Dates, Runtime, Status)
    'title',
    'original_language', # ISO code
    'overview',
    'tagline',
    'release_date',
    'release_year',
    'release_month',
    'release_decade',
    'movie_age',
    'is_recent',
    'runtime',
    'runtime_bin',
    'is_short_film',
    'is_feature_film',
    'is_long_film',
    'status',
    'adult', # Boolean flag
    'homepage', # Website link

    # Image / Collection
    'poster_url',  # Derived full URL
    'collection_name', # Derived collection name (raw belongs_to_collection is not in this list)

    # Financial
    'budget',
    'revenue',
    'profit', # Derived
    'roi',    # Derived
    'log_budget', # Derived
    'log_revenue', # Derived
    'log_profit', # Derived

    # Ratings / Popularity
    'vote_average',       # Primary/Merged average
    'vote_count',         # Primary/Merged count
    'vote_min',           # Derived
    'vote_max',           # Derived
    'popularity',         # Raw TMDB popularity
    'popularity_score',   # Derived popularity?
    'critical_success',   # Derived
    'crowd_approval',     # Derived

    # Genres
    'genre_list',   # List of genre names
    'main_genre',   # Primary genre
    'genre_count',  # Derived count
    # (Note: genres_x, genres_y, etc. are not in the list you provided)

    # Keywords / Tags
    'keywords',       # TMDB Keywords (list of dicts)
    'keyword_count',  # Derived count
    'tag',            # MovieLens Tags (list?)
    'tag_count',      # Derived count

    # People (Credits)
    'director',
    'director_popularity', # Derived
    'producer',
    'producer_popularity', # Derived
    'lead_actor',
    'lead_actor_popularity', # Derived
    'lead_actor_genre_diversity', # Derived
    'other_lead',
    'other_lead_popularity', # Derived
    'other_lead_genre_diversity', # Derived
    'other_actors',   # List of other actors

    # Production Details (Language, Country, Company)
    'spoken_language_names',   # List of names (spoken_languages list of dicts is not in this list)
    'main_language',           # Derived primary
    'has_translation',         # Derived flag
    'production_country_names',# List of names (production_countries list of dicts is not in this list)
    'main_country',            # Derived primary
    'production_company_names',# List of names (production_companies list of dicts is not in this list)
    'main_production_company', # Derived primary
    'company_popularity',      # Derived
    'franchise_strength',      # Derived

    # (Note: video and imdb_id added to IDs/Core groups as they were in your list)
]

# You can use this list to select and reorder columns in your pandas DataFrame:
movies_full = movies_full[final_column_order]

In [ ]:
movies_full

In [ ]:
[col for col in movies_full.columns if "_tmdb" in col]

In [ ]:
[x for x in movies_full.columns if "tag" in x]

In [ ]:
movies_full.shape

In [ ]:
movies_full.to_parquet("movies_data_updated.parquet", index = False)